In [1]:
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 2000)

In [ ]:
import json
from datasets import Dataset, DatasetDict, concatenate_datasets, Value, Sequence, load_dataset
from huggingface_hub import login
import random

login()  # if you haven't already

In [3]:
with open("./data/raw/data.json", "r") as f:
    raw = json.load(f)

In [8]:
# make the proof dataset
examples = []

for ex in raw.values():   # however you're iterating
    gi, gl = ex["graph_features"]

    nodes = []
    for idx, meta in ex["x"]:
        nodes.append({
            "node_index": int(idx),
            "num": int(meta["num"]),
            "label": meta["label"],
            "statement": meta["statement"],
        })

    examples.append({
        "theorem_index": int(gi),
        "theorem_label": gl,
        "graph_type": "proof",
        "nodes": nodes,                     # renamed from "x"
        "edge_index": ex["edge_index"],     # list of [2, num_edges]
        "edge_attr": ex["edge_attr"],       # same
    })

pf_ds = Dataset.from_list(examples)

In [9]:
# make the theorem dataset
examples = []

for index, ex in enumerate(raw.values()):
    gi, gl = ex["graph_features"]    # theorem index, theorem label
    x_list = ex["x"]                 # list: [ [idx, meta_dict], ... ]

    # 1. Hypothesis metas (meta["label"] == "$e")
    hyp_metas = []
    for pf_idx, meta in x_list:
        if meta["label"] == "$e":
            hyp_metas.append(meta)

    # 2. Conclusion meta: last step in x_list
    conc_meta = x_list[-1][1]   # the meta-dict
    theorem_label = gl          # EXACT string, including "nan" (real theorem)

    nodes = []
    node_index = 0

    # Build hypothesis nodes
    for meta in hyp_metas:
        nodes.append({
            "node_index": node_index,
            "num": int(meta["num"]),
            "label": meta["label"],          # "$e"
            "statement": meta["statement"],
        })
        node_index += 1

    # Build conclusion node
    nodes.append({
        "node_index": node_index,
        "num": int(conc_meta["num"]),
        "label": theorem_label,             # theorem name
        "statement": conc_meta["statement"],
    })

    # 3. Edge index: hypotheses -> conclusion
    num_hyps = len(hyp_metas)
    conclusion_idx = node_index

    if num_hyps > 0:
        edge_index = [
            list(range(num_hyps)),           # sources
            [conclusion_idx] * num_hyps,     # targets
        ]
    else:
        edge_index = [[], []]

    examples.append({
        "theorem_index": int(gi),
        "theorem_label": theorem_label,
        "graph_type": "theorem",
        "nodes": nodes,
        "edge_index": edge_index,
        "edge_attr": [[]],    # you can include or drop this
    })

# Convert to HF Dataset
thm_ds = Dataset.from_list(examples)

In [10]:
# deal with empty edge_attr in thm_ds
thm_features = thm_ds.features.copy()
thm_features["edge_attr"] = Sequence(Sequence(Value("int64")))

thm_ds = thm_ds.cast(thm_features)

Casting the dataset:   0%|          | 0/42494 [00:00<?, ? examples/s]

In [11]:
#combine thm and pf datasets
combined_ds = concatenate_datasets([thm_ds, pf_ds])

In [18]:
# make train/val/test

# set seed for random # generation
random.seed(10)

n_thm = len(thm_ds)
n_pf = len(pf_ds)
assert n_thm == n_pf, f"Expected equal counts, got {n_thm} theorems and {n_pf} proofs"

# In combined_ds = concatenate_datasets([thm_ds, pf_ds])
# theorem indices: 0 .. n_thm-1
# proof indices:   n_thm .. n_thm + n_pf - 1
proof_indices = list(range(n_thm, n_thm + n_pf))

# 80% train, 10% val, 10% test on proofs
n_train_pf = int(n_pf * 0.8)
n_val_pf   = int(n_pf * 0.1)
n_test_pf  = n_pf - n_train_pf - n_val_pf  # whatever is left

train_pf = random.sample(proof_indices, n_train_pf)
remaining = sorted(set(proof_indices) - set(train_pf))
val_pf   = random.sample(remaining, n_val_pf)
remaining2 = sorted(set(remaining) - set(val_pf))
test_pf  = remaining2   # length should be n_test_pf

# All theorems go in train
train_indices = list(range(n_thm)) + sorted(train_pf)
val_indices   = sorted(val_pf)
test_indices  = sorted(test_pf)

# Safety checks
assert len(set(train_indices) & set(val_indices)) == 0
assert len(set(train_indices) & set(test_indices)) == 0
assert len(set(val_indices) & set(test_indices)) == 0
assert len(proof_indices) == len(train_pf) + len(val_pf) + len(test_pf)

In [24]:
# train/validation/test splits
train_ds = combined_ds.select(train_indices)
val_ds   = combined_ds.select(val_indices)
test_ds  = combined_ds.select(test_indices)

In [ ]:
#upload raw to HF
dataset = DatasetDict({"train":train_ds, "validation":val_ds, "test":test_ds})
dataset.push_to_hub("jableable/metamath-proof-graphs",config_name="raw")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.73k [00:00<?, ?B/s]

c:\Users\Jared\anaconda3\envs\dataset_env\lib\site-packages\huggingface_hub\file_download.py:121: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Jared\.cache\huggingface\hub\datasets--jableable--metamath-proof-graphs. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


CommitInfo(commit_url='https://huggingface.co/datasets/jableable/metamath-proof-graphs/commit/a03801c7f9d5e035674f530261de97429c72953e', commit_message='Upload dataset', commit_description='', oid='a03801c7f9d5e035674f530261de97429c72953e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/jableable/metamath-proof-graphs', endpoint='https://huggingface.co', repo_type='dataset', repo_id='jableable/metamath-proof-graphs'), pr_revision=None, pr_num=None)

In [ ]:
from datasets import load_dataset

ds = load_dataset("jableable/metamath-proof-graphs")

README.md:   0%|          | 0.00/5.51k [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/7.18M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/82.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/84988 [00:00<?, ? examples/s]

: 

In [ ]:
theorem_graphs = ds.filter(lambda ex: ex["graph_type"] == "theorem")

Filter:   0%|          | 0/84988 [00:00<?, ? examples/s]

: 

In [ ]:
len(theorem_graphs['train'])

42494

: 

In [ ]:
proof_graphs = ds.filter(lambda ex: ex["graph_type"] == "proof")

Filter:   0%|          | 0/84988 [00:00<?, ? examples/s]

: 

In [ ]:
len(proof_graphs['train'])

42494

: 

In [ ]:
theorem_graphs['train'][824]

{'theorem_index': 824,
 'theorem_label': 'syl12anc',
 'graph_type': 'theorem',
 'nodes': [{'label': '$e',
   'node_index': 0,
   'num': 7,
   'statement': '( ph -> ps )'},
  {'label': '$e', 'node_index': 1, 'num': 11, 'statement': '( ph -> ch )'},
  {'label': '$e', 'node_index': 2, 'num': 12, 'statement': '( ph -> th )'},
  {'label': '$e',
   'node_index': 3,
   'num': 14,
   'statement': '( ( ps /\\ ( ch /\\ th ) ) -> ta )'},
  {'label': 'syl12anc',
   'node_index': 4,
   'num': 15,
   'statement': '( ph -> ta )'}],
 'edge_index': [[0, 1, 2, 3], [4, 4, 4, 4]],
 'edge_attr': [[]]}

: 

In [ ]:
proof_graphs['train'][816]

{'theorem_index': 816,
 'theorem_label': 'impimprbi',
 'graph_type': 'proof',
 'nodes': [{'label': 'dfbi2',
   'node_index': 0,
   'num': 30,
   'statement': '( ( ph <-> ps ) <-> ( ( ph -> ps ) /\\ ( ps -> ph ) ) )'},
  {'label': 'pm5.1',
   'node_index': 1,
   'num': 37,
   'statement': '( ( ( ph -> ps ) /\\ ( ps -> ph ) ) -> ( ( ph -> ps ) <-> ( ps -> ph ) ) )'},
  {'label': 'sylbi',
   'node_index': 2,
   'num': 38,
   'statement': '( ( ph <-> ps ) -> ( ( ph -> ps ) <-> ( ps -> ph ) ) )'},
  {'label': 'impbi',
   'node_index': 3,
   'num': 50,
   'statement': '( ( ph -> ps ) -> ( ( ps -> ph ) -> ( ph <-> ps ) ) )'},
  {'label': 'pm2.521',
   'node_index': 4,
   'num': 63,
   'statement': '( -. ( ph -> ps ) -> ( ps -> ph ) )'},
  {'label': 'pm2.24d',
   'node_index': 5,
   'num': 64,
   'statement': '( -. ( ph -> ps ) -> ( -. ( ps -> ph ) -> ( ph <-> ps ) ) )'},
  {'label': 'bija',
   'node_index': 6,
   'num': 65,
   'statement': '( ( ( ph -> ps ) <-> ( ps -> ph ) ) -> ( ph <-> ps )

: 

In [ ]:
train = ds["train"]

: 

In [ ]:
example = train[817]
example

{'theorem_index': 817,
 'theorem_label': 'nan',
 'nodes': [{'label': 'impexp',
   'node_index': 0,
   'num': 23,
   'statement': '( ( ( ph /\\ ps ) -> -. ch ) <-> ( ph -> ( ps -> -. ch ) ) )'},
  {'label': 'imnan',
   'node_index': 1,
   'num': 35,
   'statement': '( ( ps -> -. ch ) <-> -. ( ps /\\ ch ) )'},
  {'label': 'imbi2i',
   'node_index': 2,
   'num': 36,
   'statement': '( ( ph -> ( ps -> -. ch ) ) <-> ( ph -> -. ( ps /\\ ch ) ) )'},
  {'label': 'bitr2i',
   'node_index': 3,
   'num': 37,
   'statement': '( ( ph -> -. ( ps /\\ ch ) ) <-> ( ( ph /\\ ps ) -> -. ch ) )'}],
 'edge_index': [[1, 0, 2], [2, 3, 3]],
 'edge_attr': [[35, 36], [23, 37], [36, 37]]}

: 

In [ ]:
# normalize input to list
if isinstance(raw, dict):
    raw = list(raw.values())

examples = []
for ex in raw:
    gi, gl = ex["graph_features"]
    examples.append({
        "theorem_index": gi,
        "theorem_label": gl,
        "nodes": ex["x"],
        "edge_index": ex["edge_index"],
        "edge_attr": ex["edge_attr"],
    })

ds = Dataset.from_list(examples)

dataset = DatasetDict({"train": ds})
dataset.push_to_hub("jableable/metamath-proof-graphs")

ArrowInvalid: cannot mix struct and non-struct, non-null values

: 

In [ ]:
pf_data[2]

graph_features                                             [2, mp2]
x                 [[0, {'num': 3, 'label': '$e', 'statement': 'p...
edge_index                             [[1, 2, 0, 3], [3, 3, 4, 4]]
edge_attr                     [[8, 10], [9, 10], [3, 11], [10, 11]]
Name: 2, dtype: object

: 

In [ ]:
import pandas as pd
file_limit = 100000

pf_data = pd.read_json("./data/raw/data.json")
pf_data = pf_data.iloc[:, : file_limit]

: 

In [ ]:
pf_data = pd.read_json("./data/raw/data.json")

: 

In [ ]:
pf_data[817]

graph_features                                                                                                                                                                                                                                                                                                                                                                                                                                         [817, nan]
x                 [[0, {'num': 23, 'label': 'impexp', 'statement': '( ( ( ph /\ ps ) -> -. ch ) <-> ( ph -> ( ps -> -. ch ) ) )'}], [1, {'num': 35, 'label': 'imnan', 'statement': '( ( ps -> -. ch ) <-> -. ( ps /\ ch ) )'}], [2, {'num': 36, 'label': 'imbi2i', 'statement': '( ( ph -> ( ps -> -. ch ) ) <-> ( ph -> -. ( ps /\ ch ) ) )'}], [3, {'num': 37, 'label': 'bitr2i', 'statement': '( ( ph -> -. ( ps /\ ch ) ) <-> ( ( ph /\ ps ) -> -. ch ) )'}]]
edge_index                                                                                          

: 

In [ ]:
pf_data[2].edge_attr

[[8, 10], [9, 10], [3, 11], [10, 11]]

: 

In [ ]:
pf_data[2]

graph_features                                             [2, mp2]
x                 [[0, {'num': 3, 'label': '$e', 'statement': 'p...
edge_index                             [[1, 2, 0, 3], [3, 3, 4, 4]]
edge_attr                     [[8, 10], [9, 10], [3, 11], [10, 11]]
Name: 2, dtype: object

: 

In [ ]:
def find_none_like(obj, path=""):
    hits = []

    if isinstance(obj, dict):
        for k, v in obj.items():
            hits.extend(find_none_like(v, f"{path}.{k}" if path else str(k)))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            hits.extend(find_none_like(v, f"{path}[{i}]"))
    else:
        if obj is None:
            hits.append(path)
        #if obj == 'nan':
            #hits.append(path)

    return hits


all_hits = []

for col in pf_data.columns:
    structure = pf_data[col].to_dict()  # dict: row_name -> value
    hits = find_none_like(structure, f"pf_data[{col}]")
    all_hits.extend(hits)

for h in all_hits:
    print("•", h)


: 

In [ ]:
pf_data[8718]

graph_features                                    [8718, wemaplem2]
x                 [[0, {'num': 154, 'label': '$e', 'statement': ...
edge_index        [[0, 1, 3, 22, 39, 47, 72, 6, 7, 64, 65, 5, 8,...
edge_attr         [[154, 156], [155, 156], [285, 286], [680, 681...
Name: 8718, dtype: object

: 

In [ ]:
pf_data[8719]

graph_features                                    [8719, wemaplem3]
x                 [[0, {'num': 110, 'label': '$e', 'statement': ...
edge_index        [[3, 10, 1, 2, 4, 8, 9, 11, 0, 5, 7, 12, 15, 1...
edge_attr         [[184, 185], [300, 301], [162, 186], [163, 186...
Name: 8719, dtype: object

: 